In [1]:
#pip install dash_colorscales

In [2]:
import pandas as pd
import numpy as np
import dash
import plotly.express as px
from dash import dcc
from dash import html
from dash.dependencies import Input, Output, State
import dash_colorscales
import cufflinks as cf
import re

In [3]:
ufo_df = pd.read_csv('scrubbed.csv.zip', low_memory=False)

In [4]:
ufo_df.dropna(inplace=True)
ufo_df = ufo_df[ufo_df['country'] == 'us']

In [5]:
new_data = pd.read_excel('States.xlsx', engine='openpyxl')

In [6]:
def map_abb(state):
    for row in new_data.index:
        if state == new_data['ABB_lower'][row]:
            return new_data['ABB_upper'][row]
        
new_data['test'] = new_data['ABB_lower'].apply(map_abb)

In [7]:
ufo_df['state'] = ufo_df['state'].apply(map_abb)

In [8]:
ufo_df['datetime'] = ufo_df['datetime'].apply(lambda x: x[-10:-6])

In [9]:
ufo_df['datetime'] = ufo_df['datetime'].apply(lambda x: int(x))

In [10]:
ufo_df

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,1949,san marcos,TX,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
3,1956,edna,TX,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,1960,kaneohe,HI,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611
5,1961,bristol,TN,us,sphere,300,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.5950000,-82.188889
7,1965,norwalk,CT,us,disk,1200,20 minutes,A bright orange color changing to reddish colo...,10/2/1999,41.1175000,-73.408333
...,...,...,...,...,...,...,...,...,...,...,...
80327,2013,nashville,TN,us,light,600,10 minutes,Round from the distance/slowly changing colors...,9/30/2013,36.1658333,-86.784444
80328,2013,boise,ID,us,circle,1200,20 minutes,Boise&#44 ID&#44 spherical&#44 20 min&#44 10 r...,9/30/2013,43.6136111,-116.202500
80329,2013,napa,CA,us,other,1200,hour,Napa UFO&#44,9/30/2013,38.2972222,-122.284444
80330,2013,vienna,VA,us,circle,5,5 seconds,Saw a five gold lit cicular craft moving fastl...,9/30/2013,38.9011111,-77.265556


In [11]:
ufo_df['datetime'].max()

2014

In [12]:
def bin_decade(year):
    if year >= 1910 and year <= 1920:
        return "1910-1920"
    elif year >= 1920 and year <= 1930:
        return "1920-1930"
    elif year >= 1930 and year <= 1940:
        return "1930-1940"
    elif year >= 1940 and year <= 1950:
        return "1940-1950"
    elif year >= 1950 and year <= 1960:
        return "1950-1960"
    elif year >= 1960 and year <= 1970:
        return "1960-1970"
    elif year >= 1970 and year <= 1980:
        return "1970-1980"
    elif year >= 1980 and year <= 1990:
        return "1980-1990"
    elif year >= 1990 and year <= 2000:
        return "1990-2000" 
    elif year >= 2000 and year <= 2010:
        return "2000-2010"
    elif year >= 2010 and year <= 2014:
        return "2010-2014"
        

In [13]:
ufo_df['decade'] = ufo_df['datetime'].apply(bin_decade)

In [14]:
ufo_df = ufo_df[ufo_df['datetime'] >= 1950]

In [15]:
ufo_df['sighting'] = 1

In [16]:
ufo_df.to_csv('clean_df.csv')

In [17]:
ufo_df.groupby(['decade', 'state']).sum()['sighting']

decade     state
1940-1950  CA         2
           CO         2
           FL         1
           ID         1
           KY         2
                   ... 
2010-2014  VT       134
           WA       996
           WI       425
           WV       148
           WY        44
Name: sighting, Length: 371, dtype: int64

In [18]:
ufo_df.to_csv('clean_df.csv')

In [19]:
ufo_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 63464 entries, 3 to 80331
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              63464 non-null  int64  
 1   city                  63464 non-null  object 
 2   state                 63457 non-null  object 
 3   country               63464 non-null  object 
 4   shape                 63464 non-null  object 
 5   duration (seconds)    63464 non-null  object 
 6   duration (hours/min)  63464 non-null  object 
 7   comments              63464 non-null  object 
 8   date posted           63464 non-null  object 
 9   latitude              63464 non-null  object 
 10  longitude             63464 non-null  float64
 11  decade                63464 non-null  object 
 12  sighting              63464 non-null  int64  
dtypes: float64(1), int64(2), object(10)
memory usage: 6.8+ MB


In [20]:
test = pd.read_csv('clean_df.csv', index_col=0)

In [21]:
another_test = test.groupby(['decade', 'state']).sum()

In [22]:
another_test.groupby('decade').sum()

,datetime,duration (seconds),latitude,longitude,sighting
decade,,,,,
1940-1950,40950,8.037500e+03,8.192861e+02,-1.949459e+03,21
1950-1960,714040,1.607524e+06,1.423191e+04,-3.440982e+04,365
1960-1970,2164898,1.705875e+07,4.259154e+04,-1.018791e+05,1101
1970-1980,3774043,3.565447e+06,7.337317e+04,-1.764893e+05,1910
1980-1990,3145703,1.340054e+07,6.064030e+04,-1.484217e+05,1584
1990-2000,16901779,9.764115e+07,3.264803e+05,-8.288289e+05,8460
2000-2010,63215332,1.307307e+08,1.202425e+06,-3.033730e+06,31515
2010-2014,37229535,8.644529e+07,7.112173e+05,-1.739368e+06,18501


In [23]:
test = test[test['state'] == 'TX']

In [24]:
new_test = test.groupby(['decade', 'state']).sum()

In [25]:
new_test['avg_encounter'] = new_test['duration (seconds)'] / new_test['sighting']

In [26]:
new_test.reset_index(inplace=True)

In [27]:
new_test

,decade,state,datetime,duration (seconds),latitude,longitude,sighting,avg_encounter
0,1940-1950,TX,1950,600.00,29.301111,-94.797500,1,600.000000
1,1950-1960,TX,62593,35978.00,1000.959722,-3136.974445,32,1124.312500
2,1960-1970,TX,117978,3874280.00,1856.321945,-5897.586944,60,64571.333333
3,1970-1980,TX,237066,242926.00,3720.060279,-11717.562500,120,2024.383333
4,1980-1990,TX,204579,72615.00,3204.853890,-10079.457500,103,705.000000
5,1990-2000,TX,924968,880096.00,14387.334724,-45061.607781,463,1900.855292
6,2000-2010,TX,3793525,2251666.25,58934.376949,-184308.077231,1891,1190.727790
7,2010-2014,TX,1450773,1076014.00,22342.008613,-70194.926114,721,1492.391123


In [28]:
new_test['decade'].values

array(['1940-1950', '1950-1960', '1960-1970', '1970-1980', '1980-1990',
       '1990-2000', '2000-2010', '2010-2014'], dtype=object)

In [29]:
import plotly.express as px

In [30]:
fig = px.bar(new_test, x='decade', y='sighting')
fig.show()

In [31]:
import json
import pandas as pd
import plotly.express as px  # (version 4.7.0)
import plotly.graph_objects as go

#import dash  # (version 1.12.0) pip install dash
#import dash_core_components as dcc
#import dash_html_components as html
#from dash.dependencies import Input, Output

df = pd.read_csv("clean_df.csv", index_col=0)

colors = {
    'background': '#111111',
    'text': '#C5DB5F'
}

app = dash.Dash(__name__)

# ------------------------------------------------------------------------------
# App layout
app.layout = html.Div(style={'backgroundColor': colors['background']}, children=[

    html.H1("UFO Sightings in the U.S.", style={'text-align': 'center', 'color': colors['text']}),
    
    html.Div([
        html.Div([
            dcc.Dropdown(id="slct_decade",
                        options=[
                            {"label": "1940-1950", "value": "1940-1950"},
                            {"label": "1950-1960", "value": "1950-1960"},
                            {"label": "1960-1970", "value": "1960-1970"},
                            {"label": "1970-1980", "value": "1970-1980"},
                            {"label": "1990-2000", "value": "1990-2000"},
                            {"label": "2000-2010", "value": "2000-2010"},
                            {"label": "2010-2014", "value": "2010-2014"}],
                        multi=False,
                        value="2010-2014", 
                        style={'width': "50%"}
                        ),

            html.Div(id='output_container', style={'color': colors['text']}, children=[]),
            html.Br(),
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Dropdown(id="slct_chart",
                        options=[
                            {"label": "Average Encounter (seconds)", "value": "avg_encounter"},
                            {"label": "UFO Sightings", "value": "sighting"}],
                        multi=False,
                        value="sighting",
                        style={'width': "76"}
                        ),
            html.Br(),
        ], style={'width': '49%', 'text-align': 'center', 'display': 'inline-block'})
    ], style={
        'padding': '10px 5px'
    }),
    html.Div([
        dcc.Graph(id='my_ufo_map',
        hoverData={'points': [{'customdata': ['TX', 0]}]}
        )
    ], style={'width': '49%', 'display': 'inline-block'}),
    html.Div([
        dcc.Graph(id='my_bar_chart'),
    ], style={'display': 'inline-block', 'width': '49%'}),
])

# ------------------------------------------------------------------------------
# Connect the Plotly graphs with Dash Components
@app.callback(
    [Output(component_id='output_container', component_property='children'),
     Output(component_id='my_ufo_map', component_property='figure')],
    [Input(component_id='slct_decade', component_property='value')]
)
def update_map(slct_decade):

    container = "The decade chosen by user was: {}".format(slct_decade)
    
    dff = df.copy()
    dff = dff.groupby(['decade', 'state']).count()[['sighting']]
    dff.reset_index(inplace=True)
    dff = dff[dff["decade"] == slct_decade]

    # Plotly Express
    fig = px.choropleth(
        data_frame=dff,
        locationmode='USA-states',
        locations='state',
        scope="usa",
        color='sighting',
        hover_data=['state', 'sighting'],
        color_continuous_scale=px.colors.sequential.Blugrn,
        labels={'sighting': 'UFO sightings'},
        template='plotly_dark'
    )

    return container, fig
    
def create_chart(df_new, slct_chart, hoverData):

    df_new = df_new.groupby('decade').sum()
    df_new.reset_index(inplace=True)
    df_new['avg_encounter'] = df_new['duration (seconds)'] / df_new['sighting']
    x = df_new['decade']

    fig = px.bar(df_new, x='decade', y=slct_chart, title=hoverData['points'][0]['customdata'][0], template='plotly_dark')

    fig.update_layout(title={'xanchor':'center', 'yanchor': 'top', 'y':0.9,'x':0.5,})

    return fig 

@app.callback(
    Output(component_id='my_bar_chart', component_property='figure'),
    [Input(component_id='my_ufo_map', component_property='hoverData'),
    Input(component_id='slct_chart', component_property='value')]
)

def update_chart(hoverData, slct_chart):
    state_name = hoverData['points'][0]['customdata'][0]
    df_new = df.copy()
    df_new = df_new[df_new['state'] == state_name]
    print(state_name)
    return create_chart(df_new, slct_chart, hoverData)


# ------------------------------------------------------------------------------
if __name__ == '__main__':
    app.run_server(debug=True)


In [32]:
#pip list --format=freeze > requirements.txt

TX
TX
TX
OK
UT
WY
NE
KS
GA
